<a href="https://colab.research.google.com/github/Sathyarajpalle/NLP-practice/blob/main/word2vec_and_avgword2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
messages = pd.read_csv('/content/SMSSpamCollection.txt', sep='\t', names=["label", "message"])

In [9]:
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [10]:
messages.shape

(5572, 2)

In [11]:
messages['message'].loc[100]

"Please don't text me anymore. I have nothing else to say."

In [12]:
#data cleaning and preprocessing
import re
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [13]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [14]:
corpus = []
for i in range(0,len(messages)):
  review = re.sub('[^a-zA-Z0-9]', ' ', messages['message'][i]) #we remove all the special character with a space using substitute function of re
  review = review.lower()
  review = review.split()
  review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
  review = ' '.join(review)
  corpus.append(review)

In [15]:
corpus

['go jurong point crazi avail bugi n great world la e buffet cine got amor wat',
 'ok lar joke wif u oni',
 'free entri 2 wkli comp win fa cup final tkt 21st may 2005 text fa 87121 receiv entri question std txt rate c appli 08452810075over18',
 'u dun say earli hor u c alreadi say',
 'nah think goe usf live around though',
 'freemsg hey darl 3 week word back like fun still tb ok xxx std chg send 1 50 rcv',
 'even brother like speak treat like aid patent',
 'per request mell mell oru minnaminungint nurungu vettam set callertun caller press 9 copi friend callertun',
 'winner valu network custom select receivea 900 prize reward claim call 09061701461 claim code kl341 valid 12 hour',
 'mobil 11 month u r entitl updat latest colour mobil camera free call mobil updat co free 08002986030',
 'gonna home soon want talk stuff anymor tonight k cri enough today',
 'six chanc win cash 100 20 000 pound txt csh11 send 87575 cost 150p day 6day 16 tsandc appli repli hl 4 info',
 'urgent 1 week free mem

In [16]:
#creating Bag of words model
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=2500,binary = True,ngram_range=(2,2))  #binary=True makes the numbers into binary
x = cv.fit_transform(corpus).toarray()

In [17]:
x.shape

(5572, 2500)

In [18]:
y = pd.get_dummies(messages['label']).astype(int)
y = y.iloc[:,1].values

In [19]:
y

array([0, 0, 1, ..., 0, 0, 0])

In [20]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.20,random_state=0)

In [21]:
from sklearn.naive_bayes import MultinomialNB
spam_detect_model = MultinomialNB().fit(x_train,y_train)

In [22]:
y_pred = spam_detect_model.predict(x_test)

In [23]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
accuracy = accuracy_score(y_test,y_pred)

In [24]:
accuracy

0.9713004484304932

In [25]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98       955
           1       1.00      0.80      0.89       160

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [26]:
#creating the TFIDF model
from sklearn.feature_extraction.text import TfidfVectorizer
tv = TfidfVectorizer(max_features=2500,ngram_range=(1,2))
x = tv.fit_transform(corpus).toarray()

In [27]:
x

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [28]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.20,random_state=0)

In [29]:
from sklearn.naive_bayes import MultinomialNB
spam_detect_model = MultinomialNB().fit(x_train,y_train)

In [30]:
y_pred = spam_detect_model.predict(x_test)

In [31]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
accuracy = accuracy_score(y_test,y_pred)
accuracy

0.9811659192825112

In [32]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       955
           1       1.00      0.87      0.93       160

    accuracy                           0.98      1115
   macro avg       0.99      0.93      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [33]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 48.5 MB/s eta 0:00:00


In [34]:
import gensim.downloader as api

wv = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [35]:
wv['king']

array([ 1.25976562e-01,  2.97851562e-02,  8.60595703e-03,  1.39648438e-01,
       -2.56347656e-02, -3.61328125e-02,  1.11816406e-01, -1.98242188e-01,
        5.12695312e-02,  3.63281250e-01, -2.42187500e-01, -3.02734375e-01,
       -1.77734375e-01, -2.49023438e-02, -1.67968750e-01, -1.69921875e-01,
        3.46679688e-02,  5.21850586e-03,  4.63867188e-02,  1.28906250e-01,
        1.36718750e-01,  1.12792969e-01,  5.95703125e-02,  1.36718750e-01,
        1.01074219e-01, -1.76757812e-01, -2.51953125e-01,  5.98144531e-02,
        3.41796875e-01, -3.11279297e-02,  1.04492188e-01,  6.17675781e-02,
        1.24511719e-01,  4.00390625e-01, -3.22265625e-01,  8.39843750e-02,
        3.90625000e-02,  5.85937500e-03,  7.03125000e-02,  1.72851562e-01,
        1.38671875e-01, -2.31445312e-01,  2.83203125e-01,  1.42578125e-01,
        3.41796875e-01, -2.39257812e-02, -1.09863281e-01,  3.32031250e-02,
       -5.46875000e-02,  1.53198242e-02, -1.62109375e-01,  1.58203125e-01,
       -2.59765625e-01,  

In [36]:
wv.most_similar('man')

[('woman', 0.7664012908935547),
 ('boy', 0.6824871301651001),
 ('teenager', 0.6586930155754089),
 ('teenage_girl', 0.6147903203964233),
 ('girl', 0.5921714305877686),
 ('suspected_purse_snatcher', 0.571636438369751),
 ('robber', 0.5585119128227234),
 ('Robbery_suspect', 0.5584409832954407),
 ('teen_ager', 0.5549196600914001),
 ('men', 0.5489763021469116)]

In [37]:
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()

In [38]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [39]:
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z0-9]', ' ', messages['message'][i])
    review = review.lower()
    review = review.split()

    review = [lemmatizer.lemmatize(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [40]:
corpus

['go jurong point crazy available bugis n great world la e buffet cine got amore wat',
 'ok lar joking wif u oni',
 'free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry question std txt rate c apply 08452810075over18',
 'u dun say early hor u c already say',
 'nah think go usf life around though',
 'freemsg hey darling 3 week word back like fun still tb ok xxx std chgs send 1 50 rcv',
 'even brother like speak treat like aid patent',
 'per request melle melle oru minnaminunginte nurungu vettam set callertune caller press 9 copy friend callertune',
 'winner valued network customer selected receivea 900 prize reward claim call 09061701461 claim code kl341 valid 12 hour',
 'mobile 11 month u r entitled update latest colour mobile camera free call mobile update co free 08002986030',
 'gonna home soon want talk stuff anymore tonight k cried enough today',
 'six chance win cash 100 20 000 pound txt csh11 send 87575 cost 150p day 6days 16 tsandcs apply reply

In [84]:
len(corpus)

5572

In [41]:
corpus[0]

'go jurong point crazy available bugis n great world la e buffet cine got amore wat'

In [42]:
from nltk.tokenize import sent_tokenize
from gensim.utils import simple_preprocess

In [43]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [44]:
words = []
for sent in corpus:
  sent_token = sent_tokenize(sent)
  for sent in sent_token:
    words.append(simple_preprocess(sent))


In [45]:
words

[['go',
  'jurong',
  'point',
  'crazy',
  'available',
  'bugis',
  'great',
  'world',
  'la',
  'buffet',
  'cine',
  'got',
  'amore',
  'wat'],
 ['ok', 'lar', 'joking', 'wif', 'oni'],
 ['free',
  'entry',
  'wkly',
  'comp',
  'win',
  'fa',
  'cup',
  'final',
  'tkts',
  'st',
  'may',
  'text',
  'fa',
  'receive',
  'entry',
  'question',
  'std',
  'txt',
  'rate',
  'apply',
  'over'],
 ['dun', 'say', 'early', 'hor', 'already', 'say'],
 ['nah', 'think', 'go', 'usf', 'life', 'around', 'though'],
 ['freemsg',
  'hey',
  'darling',
  'week',
  'word',
  'back',
  'like',
  'fun',
  'still',
  'tb',
  'ok',
  'xxx',
  'std',
  'chgs',
  'send',
  'rcv'],
 ['even', 'brother', 'like', 'speak', 'treat', 'like', 'aid', 'patent'],
 ['per',
  'request',
  'melle',
  'melle',
  'oru',
  'minnaminunginte',
  'nurungu',
  'vettam',
  'set',
  'callertune',
  'caller',
  'press',
  'copy',
  'friend',
  'callertune'],
 ['winner',
  'valued',
  'network',
  'customer',
  'selected',
  're

In [83]:
len(words)

5565

In [46]:
import gensim

In [47]:
model = gensim.models.Word2Vec(words,
    window=5,
    min_count=2
)

In [48]:
model.wv.index_to_key

['call',
 'get',
 'ur',
 'gt',
 'lt',
 'go',
 'ok',
 'day',
 'free',
 'know',
 'come',
 'like',
 'good',
 'time',
 'got',
 'love',
 'text',
 'want',
 'send',
 'one',
 'need',
 'txt',
 'today',
 'going',
 'stop',
 'home',
 'lor',
 'sorry',
 'see',
 'still',
 'take',
 'mobile',
 'back',
 'da',
 'dont',
 'reply',
 'think',
 'tell',
 'week',
 'phone',
 'hi',
 'new',
 'later',
 'please',
 'pls',
 'co',
 'msg',
 'dear',
 'make',
 'night',
 'message',
 'well',
 'say',
 'min',
 'thing',
 'much',
 'hope',
 'oh',
 'claim',
 'great',
 'hey',
 'number',
 'give',
 'happy',
 'work',
 'friend',
 'wat',
 'way',
 'yes',
 'www',
 'let',
 'prize',
 'right',
 'tomorrow',
 'already',
 'said',
 'ask',
 'win',
 'amp',
 'cash',
 'life',
 'im',
 'yeah',
 'tone',
 'really',
 'babe',
 'meet',
 'find',
 'miss',
 'morning',
 'last',
 'service',
 'uk',
 'thanks',
 'care',
 'would',
 'anything',
 'com',
 'year',
 'also',
 'nokia',
 'lol',
 'every',
 'feel',
 'keep',
 'pick',
 'sure',
 'sent',
 'contact',
 'urgent',


In [49]:
model.wv['kid']

array([-0.06228501,  0.09508878,  0.05588546,  0.03243726,  0.02919779,
       -0.16262314,  0.02860911,  0.18622848, -0.05950994, -0.04629852,
       -0.07969293, -0.1442019 , -0.01745287,  0.05678036,  0.00185069,
       -0.08076467,  0.01241986, -0.17600662, -0.01754197, -0.20323141,
        0.03865481,  0.03905794,  0.02839207, -0.04593775, -0.03368705,
        0.00577882, -0.09646189, -0.08829075, -0.08588032,  0.01949743,
        0.13985895,  0.00921513,  0.04890451, -0.07719198, -0.03316399,
        0.09494616,  0.00426855, -0.08919137, -0.09881253, -0.17475954,
       -0.02391803, -0.09244253, -0.01259851,  0.04234757,  0.10063387,
       -0.05424536, -0.06275666, -0.02203222,  0.04597023,  0.08757667,
        0.06664044, -0.10108577, -0.0566635 , -0.01338435, -0.09965076,
        0.04143664,  0.07880496,  0.01292535, -0.13690497,  0.01618438,
        0.01567214,  0.03243044, -0.05084987,  0.00800767, -0.10989441,
        0.06119424,  0.03685071,  0.06319983, -0.13454005,  0.12

In [50]:
model.corpus_count

5565

In [51]:
model.epochs

5

In [52]:
model.wv.similar_by_word('kid')

[('da', 0.9977508783340454),
 ('going', 0.9976682662963867),
 ('sure', 0.9976599812507629),
 ('next', 0.9976325035095215),
 ('lot', 0.997628390789032),
 ('come', 0.9975988268852234),
 ('live', 0.9975954294204712),
 ('feel', 0.9975563883781433),
 ('eat', 0.9975548386573792),
 ('end', 0.9975548386573792)]

In [53]:
model.wv.similar_by_word('happy')

[('day', 0.9994271993637085),
 ('make', 0.9993947148323059),
 ('dear', 0.999392569065094),
 ('dont', 0.9993918538093567),
 ('would', 0.9993907809257507),
 ('said', 0.9993894696235657),
 ('go', 0.9993765950202942),
 ('one', 0.9993704557418823),
 ('new', 0.9993699789047241),
 ('amp', 0.9993699193000793)]

In [54]:
model.wv['kid'].shape

(100,)

In [56]:
!pip install tqdm

In [67]:
def avg_word2vec(doc):
  return np.mean([model.wv[word] for word in doc if word in model.wv.index_to_key],axis=0)

In [57]:
import numpy as np
from tqdm import tqdm

In [60]:
words[73]

['performed']

In [61]:
type(model.wv.index_to_key)

list

In [65]:
#apply for the entire sentences
X=[]
for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))

100%|██████████| 5565/5565 [00:00<00:00, 9232.76it/s] 


In [68]:
X

[array([-0.14201802,  0.22891955,  0.11901461,  0.07613007,  0.09383295,
        -0.39944828,  0.0621882 ,  0.47811195, -0.15089361, -0.1140523 ,
        -0.20988475, -0.38009992, -0.03347955,  0.12539476,  0.01997512,
        -0.21631062,  0.02066173, -0.4187024 , -0.06296708, -0.4840219 ,
         0.08667893,  0.10390178,  0.05433089, -0.12573896, -0.09261949,
         0.01140728, -0.25568673, -0.22189815, -0.21862273,  0.06701515,
         0.3288722 ,  0.01987911,  0.09652448, -0.16895431, -0.08823213,
         0.22211836, -0.00703614, -0.22668993, -0.22322273, -0.42613482,
        -0.06081383, -0.24632762, -0.04365312,  0.09963342,  0.23887403,
        -0.11624875, -0.14727746, -0.0677519 ,  0.13755833,  0.21233904,
         0.13925116, -0.27235037, -0.11930262, -0.03226812, -0.22397776,
         0.09576946,  0.18735397,  0.02738536, -0.32896897,  0.045943  ,
         0.05576181,  0.09556203, -0.10151776,  0.03214112, -0.28432363,
         0.15252289,  0.09435818,  0.16479145, -0.3

In [69]:
type(X)

list

In [75]:
x_new = np.array(X)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (5565,) + inhomogeneous part.

In [76]:
x_new[3]

array([-0.17341541,  0.27989867,  0.13928483,  0.09865641,  0.11244676,
       -0.48722041,  0.07993399,  0.57716447, -0.18214051, -0.13379014,
       -0.26067773, -0.45937166, -0.03592146,  0.15065292,  0.02311052,
       -0.26006827,  0.0260517 , -0.50418288, -0.07799017, -0.58280212,
        0.10785667,  0.12700322,  0.07105397, -0.14542167, -0.1109178 ,
        0.0155538 , -0.3063319 , -0.26583344, -0.26763302,  0.08854351,
        0.39836013,  0.02118198,  0.12103825, -0.20881015, -0.10386286,
        0.27338439, -0.01374208, -0.27454564, -0.27114773, -0.5155915 ,
       -0.0729841 , -0.29886678, -0.05044654,  0.12397987,  0.29266265,
       -0.14606641, -0.17953986, -0.08317517,  0.1667438 ,  0.2602841 ,
        0.17035842, -0.33685911, -0.14928956, -0.04072153, -0.26288614,
        0.11515471,  0.22766377,  0.03142286, -0.39885616,  0.05092202,
        0.06630752,  0.10985681, -0.1241341 ,  0.03943187, -0.34625244,
        0.18460231,  0.11302605,  0.20691796, -0.38388288,  0.39

In [78]:
words[0]

['go',
 'jurong',
 'point',
 'crazy',
 'available',
 'bugis',
 'great',
 'world',
 'la',
 'buffet',
 'cine',
 'got',
 'amore',
 'wat']

In [77]:
x_new[0].shape

(100,)

In [80]:
print(x_new.shape)
print(y.shape)

(5565, 100)
(5572,)


In [79]:
import numpy as np
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_new,y,test_size=0.20,random_state=0)

ValueError: Found input variables with inconsistent numbers of samples: [5565, 5572]